In [1]:
%cd /content
!rm -rf GeoChat
!git clone https://github.com/mbzuai-oryx/GeoChat.git
%cd GeoChat

# Install the geochat package itself without pip trying to force its ancient exact pins
# (torch==2.0.1 and tokenizers==0.13.3 have no wheels for Colab's current Python/CUDA)
!pip install -e . --no-deps

# Runtime deps the demo actually needs, at versions that exist today.
# Not touching torch — keep Colab's preinstalled CUDA-matched build for the T4.
!pip install transformers==4.36.2 accelerate bitsandbytes peft \
    sentencepiece timm einops einops-exts markdown2 shortuuid scikit-learn httpx

!pip install --no-deps gradio==3.50.2 gradio-client==0.6.1 huggingface-hub==0.20.3 ffmpy aiofiles markdown-it-py mdurl pygments
!pip install "jinja2==3.1.2" --force-reinstall --no-deps

/content
Cloning into 'GeoChat'...
remote: Enumerating objects: 480, done.
remote: Counting objects: 100% (80/80), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 480 (delta 64), reused 38 (delta 38), pack-reused 400 (from 1)
Receiving objects: 100% (480/480), 63.84 MiB | 19.97 MiB/s, done.
Resolving deltas: 100% (155/155), done.
/content/GeoChat
Obtaining file:///content/GeoChat
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for geochat (pyproject.toml) ... done
  Created wheel for geochat: filename=geochat-1.1.1-0.editable-py3-none-any.whl size=8153 sha256=0a3250ec11012125400985742ea3530507289a62abd628fbbe20a7b87bfdcf77
  Stored in directory: /tmp/pip-ephem-wheel-cache-7lqhk27s/wheels/77/9d/36/859da510aaf3ae59c029ede61d673ddefe5c407ba86194b5a1
Successfully built geochat
     ━━━━━━━━━━

In [3]:
import sys
import torch
from transformers.modeling_utils import PreTrainedModel

import transformers.models.bloom.modeling_bloom as bloom_mod
import transformers.models.opt.modeling_opt as opt_mod
import transformers.models.llama.modeling_llama as llama_mod

for mod in (bloom_mod, opt_mod, llama_mod):
    if not hasattr(mod, "_expand_mask"):
        mod._expand_mask = lambda mask, dtype, tgt_len=None: mask
    if not hasattr(mod, "_make_causal_mask"):
        mod._make_causal_mask = lambda input_ids_shape, dtype, device=None, past_key_values_length=0: torch.zeros(0)

_old_to = PreTrainedModel.to
def _new_to(self, *args, **kwargs):
    if getattr(self, "quantization_method", None) is not None:
        return self
    return _old_to(self, *args, **kwargs)
PreTrainedModel.to = _new_to

sys.path.append('/content/GeoChat')
from geochat.model.builder import load_pretrained_model
from geochat.mm_utils import process_images, tokenizer_image_token
from geochat.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN
from PIL import Image

tokenizer, model, image_processor, context_len = load_pretrained_model(
    model_path="MBZUAI/geochat-7B",
    model_base=None,
    model_name="geochat-7b",
    load_8bit=True,
    load_4bit=False,
    device="cuda"
)

/usr/local/lib/python3.13/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.13/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.13/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.13/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_no

Loading GeoChat......


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/749 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:600: UserWarning: for vision_model.embeddings.class_embedding: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  module._load_from_state_dict(*args)
/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:600: UserWarning: for vision_model.embeddings.patch_embedding.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  module._load_from_state_dict(*args)
/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:600: UserWarning: for vision_model.embeddings.posi

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [4]:
import transformers.models.llama.modeling_llama as llama_mod

if not hasattr(llama_mod, "_orig_apply_rotary_pos_emb"):
    llama_mod._orig_apply_rotary_pos_emb = llama_mod.apply_rotary_pos_emb

def _patched_apply_rotary_pos_emb(q, k, cos, sin, position_ids=None, unsqueeze_dim=1):
    device = q.device
    cos = cos.to(device)
    sin = sin.to(device)
    if position_ids is not None:
        position_ids = position_ids.to(device)
    return llama_mod._orig_apply_rotary_pos_emb(q, k, cos, sin, position_ids=position_ids, unsqueeze_dim=unsqueeze_dim)

llama_mod.apply_rotary_pos_emb = _patched_apply_rotary_pos_emb

In [5]:

import re
from PIL import Image, ImageDraw
from geochat.conversation import conv_templates, SeparatorStyle
from geochat.mm_utils import KeywordsStoppingCriteria, expand2square
from geochat.constants import (
    IMAGE_TOKEN_INDEX,
    DEFAULT_IMAGE_TOKEN,
    DEFAULT_IM_START_TOKEN,
    DEFAULT_IM_END_TOKEN,
)

# -------------------------------------------------------------------------
# Helper: Parse and denormalize 0-100 coordinates to image pixel dimensions
# -------------------------------------------------------------------------
def parse_bboxes(raw_text: str, image_width: int, image_height: int):
    """
    Parses GeoChat bounding boxes: {<x0><y0><x1><y1>} or {<x0><y0><x1><y1>|<angle>}
    and converts them from normalized [0, 100] coordinates to image pixel dimensions.
    """
    boxes = []

    # 1. Check for labeled entities e.g. <p>aircraft</p>{<10><20><30><40>}
    entity_matches = re.findall(r"<p>(.*?)</p>(.*?)(?=(?:<p>|$))", raw_text, flags=re.DOTALL)
    if entity_matches:
        for label, bbox_part in entity_matches:
            for block in re.findall(r"\{<[^}]+>\}", bbox_part):
                integers = [int(x) for x in re.findall(r"-?\d+", block)]
                if len(integers) >= 4:
                    x0, y0, x1, y1 = integers[:4]
                    angle = integers[4] if len(integers) > 4 else 0
                    boxes.append({
                        "label": label.strip(),
                        "box_2d": [
                            (x0 / 100.0) * image_width,
                            (y0 / 100.0) * image_height,
                            (x1 / 100.0) * image_width,
                            (y1 / 100.0) * image_height,
                        ],
                        "angle": angle,
                    })
        return boxes

    # 2. Check for standalone boxes {<x0><y0><x1><y1>}
    for block in re.findall(r"\{<[^}]+>\}", raw_text):
        integers = [int(x) for x in re.findall(r"-?\d+", block)]
        if len(integers) >= 4:
            x0, y0, x1, y1 = integers[:4]
            angle = integers[4] if len(integers) > 4 else 0
            boxes.append({
                "label": None,
                "box_2d": [
                    (x0 / 100.0) * image_width,
                    (y0 / 100.0) * image_height,
                    (x1 / 100.0) * image_width,
                    (y1 / 100.0) * image_height,
                ],
                "angle": angle,
            })
    return boxes


# -------------------------------------------------------------------------
# Main Predict Function
# -------------------------------------------------------------------------
def predict(
    image_path: str,
    text_prompt: str,
    task_type: str = "grounding",  # "grounding", "refer", "identify", or "vqa"
    model=model,
    tokenizer=tokenizer,
    image_processor=image_processor,
    max_new_tokens: int = 512,
):
    # 1. Load image and keep dimensions for bbox scaling
    image = Image.open(image_path).convert("RGB")
    image_width, image_height = image.size

    # 2. Pad to square and resize to 504x504 (required for GeoChat's vision tower)
    image_mean = tuple(int(x * 255) for x in image_processor.image_mean)
    padded_image = expand2square(image, image_mean)
    image_tensor = image_processor.preprocess(
        padded_image,
        crop_size={"height": 504, "width": 504},
        size={"shortest_edge": 504},
        return_tensors="pt",
    )["pixel_values"]

    # Match vision tower device and dtype (float16)
    vision_dtype = model.get_vision_tower().dtype
    image_tensor = image_tensor.to(device=model.device, dtype=vision_dtype)

    # 3. Format task instruction with GeoChat's trigger tokens
    task = task_type.lower()
    if task == "grounding" and not text_prompt.startswith("[grounding]"):
        qs = f"[grounding] {text_prompt}"
    elif task == "refer" and not text_prompt.startswith("[refer]"):
        qs = f"[refer] Give me the location of <p> {text_prompt} </p>"
    elif task == "identify" and not text_prompt.startswith("[identify]"):
        qs = f"[identify] {text_prompt}"
    else:
        qs = text_prompt

    # 4. Insert image token
    if getattr(model.config, "mm_use_im_start_end", False):
        qs = DEFAULT_IM_START_TOKEN + DEFAULT_IMAGE_TOKEN + DEFAULT_IM_END_TOKEN + "\n" + qs
    else:
        qs = DEFAULT_IMAGE_TOKEN + "\n" + qs

    # 5. Build conversation prompt with Vicuna v1.5 system message
    conv = conv_templates["llava_v1"].copy()
    conv.append_message(conv.roles[0], qs)
    conv.append_message(conv.roles[1], None)
    prompt = conv.get_prompt()

    # 6. Tokenize & set up stopping criteria on </s>
    input_ids = tokenizer_image_token(
        prompt, tokenizer, IMAGE_TOKEN_INDEX, return_tensors="pt"
    ).unsqueeze(0).to(model.device)

    stop_str = conv.sep if conv.sep_style != SeparatorStyle.TWO else conv.sep2
    stopping_criteria = KeywordsStoppingCriteria([stop_str], tokenizer, input_ids)

    # 7. Generate output
    with torch.inference_mode():
        output_ids = model.generate(
            input_ids,
            images=image_tensor,
            do_sample=False,
            temperature=0.0,
            max_new_tokens=max_new_tokens,
            use_cache=True,
            stopping_criteria=[stopping_criteria],
        )

    # 8. Decode response
    input_token_len = input_ids.shape[1]
    output_text = tokenizer.decode(
        output_ids[0, input_token_len:], skip_special_tokens=True
    ).strip()

    if output_text.endswith(stop_str):
        output_text = output_text[: -len(stop_str)].strip()

    # 9. Extract pixel-space bounding boxes
    parsed_boxes = parse_bboxes(output_text, image_width, image_height)

    return {
        "text": output_text,
        "boxes": parsed_boxes,
    }

In [11]:
image_path = "/content/sample_data/church_183.png"
# Prompt can be the object name or a query:
result = predict(image_path, "church", task_type="grounding")

print("Response:", result["text"])
print("Detected Boxes:", result["boxes"])


/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:389: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


Response: {<30><45><70><73>|<90>}
Detected Boxes: [{'label': None, 'box_2d': [180.0, 270.0, 420.0, 438.0], 'angle': 90}]


In [12]:
result = predict(
    image_path,
    "the church located in the center",
    task_type="refer"
)
print(result)

{'text': '{<28><39><72><79>|<90>}', 'boxes': [{'label': None, 'box_2d': [168.00000000000003, 234.0, 432.0, 474.0], 'angle': 90}]}


In [13]:
result = predict(
    image_path,
    "Classify the image in the following classes: Church, Beach, Dense Residential, Storage Tanks.",
    task_type="vqa"
)
print("Classification:", result["text"])

Classification: Church
